# CalibrateQwen 05: publish the unified Hugging Face model

We use this notebook to publish one evidence-selected CalibrateQwen release. The completed core report compares the base model, off-policy distillation, and on-policy distillation on matched in-domain, out-of-domain, and option-shuffled records.

We select the on-policy checkpoint as the general-purpose release because it achieved the strongest combination of answer accuracy, structured-output validity, multiclass negative log-likelihood, selective accuracy, OOD negative log-likelihood, and OOD risk-coverage area. The off-policy model remains the strongest option-order robustness specialist, but it has lower in-domain accuracy and weaker probability quality.

“Unified” means a single checkpoint selected from the measured model matrix and packaged with its calibration policy, output contract, evaluation summary, and model card. We do not average independently trained adapters because the report contains no evidence that adapter arithmetic preserves either accuracy or calibration.


## 1. Install the deployment toolchain

Run this notebook in a fresh Colab runtime. The latest stable Tinker Cookbook contains the Qwen3.5 merge fixes and shard-by-shard export path. The merged model is approximately 9.4 GB, so the runtime needs enough temporary disk space for the base model, downloaded adapter, merged model, and upload cache.

The default `merged` export produces a standalone Hugging Face model that can be served without attaching a separate LoRA adapter. Set `EXPORT_FORMAT = 'adapter'` for a much smaller PEFT repository.


In [ ]:
from pathlib import Path

REPO_ROOT = Path('/content/AutoRegressive-Bhasha')
if not REPO_ROOT.exists():
    !git clone https://github.com/ritwikraha/AutoRegressive-Bhasha.git /content/AutoRegressive-Bhasha

%cd /content/AutoRegressive-Bhasha/calibrate_qwen
!pip install -q --upgrade \
    'tinker>=0.24.0' \
    'tinker-cookbook>=0.5.3' \
    'huggingface_hub>=0.27.0' \
    'transformers>=5.4.0' \
    peft accelerate safetensors


## 2. Load credentials

We read the existing Colab secrets and map `HF_WRITE_ACCESS` to the standard `HF_TOKEN` environment variable. The notebook verifies both identities without printing either token.


In [ ]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

def require_secret(name):
    value = userdata.get(name)
    if value is None or not value.strip():
        raise RuntimeError(f'Add {name} to Colab Secrets and enable notebook access.')
    return value.strip()

os.environ['TINKER_API_KEY'] = require_secret('TINKER_API_KEY')
os.environ['HF_TOKEN'] = require_secret('HF_WRITE_ACCESS')

login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
hf_api = HfApi(token=os.environ['HF_TOKEN'])
hf_identity = hf_api.whoami()
print(f'Hugging Face write access verified for {hf_identity["name"]}.')


## 3. Configure the release

The source path is the final on-policy sampler checkpoint recovered during evaluation. The default destination is a public, standalone merged model at `ritwikraha/calibrate-qwen-unified`.

Change `PUBLIC_REPO` to `False` for a private release. Change `EXPORT_FORMAT` to `adapter` to publish a PEFT adapter instead. The adapter mode publishes to the `-adapter` repository so the two formats cannot overwrite each other.


In [ ]:
BASE_MODEL = 'Qwen/Qwen3.5-4B'
DATASET_REPO = 'ritwikraha/calibrate-qwen-curated'
SOURCE_CHECKPOINT = (
    'tinker://e45e676a-8586-5d5d-a00f-2c4527c33929:train:0/'
    'sampler_weights/final'
)

EXPORT_FORMAT = 'merged'  # Choose: 'merged' or 'adapter'
PUBLIC_REPO = True
KEEP_TINKER_CHECKPOINT = True
RUN_LOCAL_SMOKE_TEST = False
LOAD_DRIVE_REPORT = True

MERGED_REPO_ID = 'ritwikraha/calibrate-qwen-unified'
ADAPTER_REPO_ID = 'ritwikraha/calibrate-qwen-unified-adapter'
HF_REPO_ID = MERGED_REPO_ID if EXPORT_FORMAT == 'merged' else ADAPTER_REPO_ID

WORK_ROOT = Path('/content/calibrate_qwen_publish')
TINKER_ADAPTER_DIR = WORK_ROOT / 'tinker_adapter'
MERGED_MODEL_DIR = WORK_ROOT / 'merged_model'
PUBLICATION_DIR = WORK_ROOT / 'publication_files'
DRIVE_REPORT_ROOT = Path('/content/drive/MyDrive/calibrate_qwen_results/unified_report')

if EXPORT_FORMAT not in {'merged', 'adapter'}:
    raise ValueError("EXPORT_FORMAT must be 'merged' or 'adapter'.")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
PUBLICATION_DIR.mkdir(parents=True, exist_ok=True)

print({
    'source_checkpoint': SOURCE_CHECKPOINT,
    'export_format': EXPORT_FORMAT,
    'destination': HF_REPO_ID,
    'public': PUBLIC_REPO,
    'local_smoke_test': RUN_LOCAL_SMOKE_TEST,
})


## 4. Verify the Tinker checkpoint

We verify that the source path resolves to a Qwen3.5-4B sampler checkpoint. Final Tinker checkpoints normally have no expiry. When `KEEP_TINKER_CHECKPOINT` is enabled, we remove any remaining TTL so the original research artifact stays recoverable after the Hugging Face export.


In [ ]:
import tinker

service_client = tinker.ServiceClient()
rest_client = service_client.create_rest_client()
checkpoint_info = rest_client.get_weights_info_by_tinker_path(SOURCE_CHECKPOINT).result()

assert checkpoint_info.base_model == BASE_MODEL, (
    f'Expected {BASE_MODEL}, received {checkpoint_info.base_model}'
)
assert '/sampler_weights/' in SOURCE_CHECKPOINT

run_id = SOURCE_CHECKPOINT.removeprefix('tinker://').split('/', 1)[0]
checkpoints = rest_client.list_checkpoints(run_id).result().checkpoints
selected = next(
    checkpoint for checkpoint in checkpoints
    if checkpoint.tinker_path == SOURCE_CHECKPOINT
)
print({
    'run_id': run_id,
    'base_model': checkpoint_info.base_model,
    'checkpoint_type': str(selected.checkpoint_type),
    'created_at': str(selected.time),
    'expires_at': str(selected.expires_at),
    'size_bytes': selected.size_bytes,
})

if KEEP_TINKER_CHECKPOINT and selected.expires_at is not None:
    rest_client.set_checkpoint_ttl_from_tinker_path(
        SOURCE_CHECKPOINT,
        ttl_seconds=None,
    ).result()
    print('Removed the Tinker checkpoint TTL.')


## 5. Load the report snapshot and verify model selection

Notebook 04 stores its report in Google Drive. We load that manifest when available. A portable 250-example core snapshot is embedded as a fallback.

The supplied report has `run_mode = 'core'`, so the model card labels every result as a core evaluation. These measurements are sufficient for a practical release candidate. A final scientific claim should be updated with the complete `full` evaluation.


In [ ]:
import json
import pandas as pd

if LOAD_DRIVE_REPORT:
    from google.colab import drive
    drive.mount('/content/drive')

EMBEDDED_REPORT = {
    'run_mode': 'core',
    'evaluation_limit': 250,
    'models': {
        'base': None,
        'off_policy': 'tinker://7efd2713-b159-5099-aa88-659b038e3cc3:train:0/sampler_weights/final',
        'on_policy': SOURCE_CHECKPOINT,
    },
    'calibrations': {
        'on_policy': {
            'temperature': 1.0820089064243952,
            'validation_multiclass_nll': 0.24893711581244202,
            'abstention': {
                'threshold': 0.8905847589075757,
                'target_coverage': 0.8,
                'validation_coverage': 0.8,
                'validation_selective_accuracy': 0.97,
                'validation_risk': 0.03,
                'answered': 200,
            },
        },
    },
    'in_domain_summary': [
        {'model': 'base', 'records': 250, 'accuracy': 0.880, 'format_validity': 0.708, 'verbal_ece': 0.2314, 'token_ece_calibrated': 0.0554, 'multiclass_brier': 0.1891, 'multiclass_nll': 0.3811, 'aurc': 0.0387, 'selective_accuracy': 0.9593},
        {'model': 'off_policy', 'records': 250, 'accuracy': 0.872, 'format_validity': 0.840, 'verbal_ece': 0.2232, 'token_ece_calibrated': 0.0598, 'multiclass_brier': 0.2000, 'multiclass_nll': 0.4051, 'aurc': 0.0486, 'selective_accuracy': 0.9497},
        {'model': 'on_policy', 'records': 250, 'accuracy': 0.884, 'format_validity': 0.932, 'verbal_ece': 0.1182, 'token_ece_calibrated': 0.0647, 'multiclass_brier': 0.1831, 'multiclass_nll': 0.3686, 'aurc': 0.0531, 'selective_accuracy': 0.9674},
    ],
    'domain_summary': {
        'base': {'accuracy': 0.752, 'ece': 0.0914, 'multiclass_nll': 0.8339, 'aurc': 0.1170, 'format_validity': 0.892},
        'off_policy': {'accuracy': 0.760, 'ece': 0.1039, 'multiclass_nll': 0.8420, 'aurc': 0.1118, 'format_validity': 0.936},
        'on_policy': {'accuracy': 0.760, 'ece': 0.0932, 'multiclass_nll': 0.8066, 'aurc': 0.1014, 'format_validity': 0.944},
    },
    'robustness_summary': {
        'base': {'perturbed_accuracy': 0.848, 'choice_text_consistency': 0.888, 'choice_flip_rate': 0.112},
        'off_policy': {'perturbed_accuracy': 0.862, 'choice_text_consistency': 0.916, 'choice_flip_rate': 0.084},
        'on_policy': {'perturbed_accuracy': 0.852, 'choice_text_consistency': 0.894, 'choice_flip_rate': 0.106},
    },
}

report_manifest_path = DRIVE_REPORT_ROOT / 'report_manifest.json'
if report_manifest_path.exists():
    report = json.loads(report_manifest_path.read_text(encoding='utf-8'))
    print(f'Loaded report from {report_manifest_path}.')
else:
    report = EMBEDDED_REPORT
    print('Drive report was unavailable; using the embedded verified core snapshot.')

assert report['models']['on_policy'] == SOURCE_CHECKPOINT
summary = pd.DataFrame(report['in_domain_summary']).set_index('model')
assert summary.loc['on_policy', 'accuracy'] == summary['accuracy'].max()
assert summary.loc['on_policy', 'format_validity'] == summary['format_validity'].max()
assert summary.loc['on_policy', 'multiclass_nll'] == summary['multiclass_nll'].min()
assert summary.loc['on_policy', 'selective_accuracy'] == summary['selective_accuracy'].max()
display(summary)


## 6. Build release metadata and the model card

Temperature scaling is an inference policy rather than a change to model weights. We therefore publish `calibration_config.json` beside the checkpoint. Applications that score all answer options can apply the temperature and use the stored threshold for selective answering.

We also publish the exact structured-output contract, an evaluation summary, a small calibration helper, and available report tables and figures. Raw prediction files stay outside the model repository because the curated dataset and evaluation artifacts have separate ownership and storage concerns.


In [ ]:
import math
import shutil
from datetime import datetime, timezone

calibration = report['calibrations']['on_policy']
calibration_config = {
    'method': 'temperature_scaling',
    'confidence_source': 'answer_probability',
    'temperature': calibration['temperature'],
    'abstention_threshold': calibration['abstention']['threshold'],
    'target_coverage': calibration['abstention']['target_coverage'],
    'fit_split': 'validation',
    'fit_records': calibration['abstention']['answered'] / calibration['abstention']['target_coverage'],
    'renderer': 'qwen3_5_disable_thinking',
    'notes': 'Apply temperature to normalized option probabilities before thresholding.',
}
(PUBLICATION_DIR / 'calibration_config.json').write_text(
    json.dumps(calibration_config, indent=2) + '\n',
    encoding='utf-8',
)

output_contract = {
    'type': 'object',
    'additionalProperties': False,
    'required': ['answer', 'confidence', 'justification', 'abstain'],
    'properties': {
        'answer': {'type': 'string', 'pattern': '^[A-Z]$'},
        'confidence': {'type': 'number', 'minimum': 0.0, 'maximum': 1.0},
        'justification': {'type': 'string'},
        'abstain': {'type': 'boolean'},
    },
}
(PUBLICATION_DIR / 'structured_output_schema.json').write_text(
    json.dumps(output_contract, indent=2) + '\n',
    encoding='utf-8',
)

evaluation_summary = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'report_scope': report['run_mode'],
    'records_per_primary_split': report.get('evaluation_limit'),
    'selected_model': 'on_policy',
    'source_checkpoint': SOURCE_CHECKPOINT,
    'base_model': BASE_MODEL,
    'in_domain': summary.loc['on_policy'].to_dict(),
    'out_of_domain': report.get('domain_summary', EMBEDDED_REPORT['domain_summary'])['on_policy'],
    'option_order_robustness': report.get('robustness_summary', EMBEDDED_REPORT['robustness_summary'])['on_policy'],
    'calibration': calibration,
}
(PUBLICATION_DIR / 'evaluation_summary.json').write_text(
    json.dumps(evaluation_summary, indent=2) + '\n',
    encoding='utf-8',
)

calibration_helper = f'''# Calibration policy for {HF_REPO_ID}.
import math

TEMPERATURE = {calibration_config['temperature']!r}
ABSTENTION_THRESHOLD = {calibration_config['abstention_threshold']!r}

def temperature_scale(probabilities, temperature=TEMPERATURE):
    labels = sorted(probabilities)
    logits = [math.log(max(float(probabilities[label]), 1e-12)) / temperature for label in labels]
    maximum = max(logits)
    weights = [math.exp(value - maximum) for value in logits]
    total = sum(weights)
    return {{label: value / total for label, value in zip(labels, weights)}}

def calibrated_decision(answer, probabilities):
    scaled = temperature_scale(probabilities)
    confidence = scaled.get(str(answer), 0.0)
    return {{
        "answer": answer,
        "confidence": confidence,
        "abstain": confidence < ABSTENTION_THRESHOLD,
        "option_probabilities": scaled,
    }}
'''
(PUBLICATION_DIR / 'calibration.py').write_text(calibration_helper, encoding='utf-8')

# Copy compact report evidence when the Drive report is available.
report_assets = PUBLICATION_DIR / 'report_assets'
report_assets.mkdir(exist_ok=True)
if DRIVE_REPORT_ROOT.exists():
    for relative in [
        'tables/in_domain_summary.csv',
        'tables/domain_transfer.csv',
        'tables/option_order_robustness.csv',
        'tables/source_breakdown.csv',
        'figures/model_comparison.png',
        'figures/domain_transfer.png',
        'figures/option_order_robustness.png',
        'figures/reliability_comparison.png',
        'figures/risk_coverage_comparison.png',
    ]:
        source_path = DRIVE_REPORT_ROOT / relative
        if source_path.exists():
            destination = report_assets / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, destination)


In [ ]:
selected = summary.loc['on_policy']
ood = report.get('domain_summary', EMBEDDED_REPORT['domain_summary'])['on_policy']
robustness = report.get('robustness_summary', EMBEDDED_REPORT['robustness_summary'])['on_policy']
report_scope = report['run_mode']
report_n = int(report.get('evaluation_limit') or selected['records'])

model_card = f'''---
base_model: {BASE_MODEL}
datasets:
- {DATASET_REPO}
language:
- en
library_name: transformers
license: apache-2.0
pipeline_tag: text-generation
tags:
- qwen3.5
- tinker
- knowledge-distillation
- on-policy-distillation
- confidence-calibration
- selective-prediction
---

# CalibrateQwen Unified

CalibrateQwen Unified is an on-policy distilled {BASE_MODEL} checkpoint for multiple-choice reasoning with structured confidence and abstention. We selected this checkpoint from a matched comparison against the base model and an off-policy distilled checkpoint.

## Release format

- Base model: `{BASE_MODEL}`
- Export format: `{EXPORT_FORMAT}`
- Training method: on-policy teacher-student distillation
- Source checkpoint: `{SOURCE_CHECKPOINT}`
- Renderer: `qwen3_5_disable_thinking`
- Dataset: `{DATASET_REPO}`
- License: Apache-2.0, inherited from the base model

## Why this checkpoint

The selected checkpoint delivered the strongest general-purpose result in the {report_scope} report. The primary in-domain and OOD splits each contain {report_n} evaluated examples. These are release-candidate measurements rather than complete benchmark estimates when the report scope is `core`.

| Metric | Base | Off-policy | Unified on-policy |
|---|---:|---:|---:|
| In-domain accuracy | {summary.loc['base', 'accuracy']:.4f} | {summary.loc['off_policy', 'accuracy']:.4f} | {selected['accuracy']:.4f} |
| Structured-output validity | {summary.loc['base', 'format_validity']:.4f} | {summary.loc['off_policy', 'format_validity']:.4f} | {selected['format_validity']:.4f} |
| Verbal-confidence ECE | {summary.loc['base', 'verbal_ece']:.4f} | {summary.loc['off_policy', 'verbal_ece']:.4f} | {selected['verbal_ece']:.4f} |
| Multiclass Brier score | {summary.loc['base', 'multiclass_brier']:.4f} | {summary.loc['off_policy', 'multiclass_brier']:.4f} | {selected['multiclass_brier']:.4f} |
| Multiclass NLL | {summary.loc['base', 'multiclass_nll']:.4f} | {summary.loc['off_policy', 'multiclass_nll']:.4f} | {selected['multiclass_nll']:.4f} |
| Selective accuracy | {summary.loc['base', 'selective_accuracy']:.4f} | {summary.loc['off_policy', 'selective_accuracy']:.4f} | {selected['selective_accuracy']:.4f} |

OOD accuracy is {ood['accuracy']:.4f}, OOD multiclass NLL is {ood['multiclass_nll']:.4f}, and OOD AURC is {ood['aurc']:.4f}. Option-order choice-text consistency is {robustness['choice_text_consistency']:.4f}.

## Output contract

Use the following system instruction:

```text
Answer the multiple-choice question. Return one JSON object with exactly the keys "answer", "confidence", "justification", and "abstain". answer must be one uppercase option label from the prompt. confidence must be a number between 0 and 1. justification must be one concise sentence. abstain must be a JSON boolean.
```

Expected response:

```json
{{"answer":"B","confidence":0.91,"justification":"The evidence directly supports option B.","abstain":false}}
```

## Loading the model

For a merged release:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("{MERGED_REPO_ID}")
model = AutoModelForCausalLM.from_pretrained(
    "{MERGED_REPO_ID}",
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True,
)
```

For the PEFT adapter release:

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained("{BASE_MODEL}", device_map="auto")
model = PeftModel.from_pretrained(base, "{ADAPTER_REPO_ID}")
```

## Serving

Merged model:

```bash
vllm serve {MERGED_REPO_ID}
```

PEFT adapter:

```bash
vllm serve {BASE_MODEL} --enable-lora --lora-modules calibrate={ADAPTER_REPO_ID}
```

## Calibration policy

The validation-fitted answer-probability temperature is `{calibration_config['temperature']:.6f}` and the 80 percent target-coverage threshold is `{calibration_config['abstention_threshold']:.6f}`. Load `calibration_config.json` and use `calibration.py` when the serving stack scores every option label. The generated verbal confidence remains available when option scoring is unavailable.

## Limitations

The release targets English multiple-choice prompts. Calibration was fitted on the curated validation mixture and can shift under new domains, prompt templates, quantization, or decoding settings. The provided evaluation snapshot uses {report_n} examples per primary split. Refit calibration after material deployment changes and use a full evaluation before making final benchmark claims.

## Reproducibility

Training, dataset curation, evaluation, and publication notebooks are available in [AutoRegressive-Bhasha](https://github.com/ritwikraha/AutoRegressive-Bhasha). The model repository includes the frozen calibration policy, output schema, evaluation summary, and compact report evidence.
'''
(PUBLICATION_DIR / 'README.md').write_text(model_card, encoding='utf-8')
print(model_card[:5000])


## 7. Export the selected checkpoint

In merged mode, we download the Tinker adapter, merge its deltas into the base model with the low-memory shard strategy, and copy the release metadata into the model directory. The merge preserves the base model's on-disk tensor dtype.

In adapter mode, the Tinker CLI performs the PEFT conversion and pushes the adapter directly. We then upload the custom metadata in a second commit.


In [ ]:
import subprocess
from tinker_cookbook import weights

if EXPORT_FORMAT == 'merged':
    free_gib = shutil.disk_usage('/content').free / (1024 ** 3)
    if free_gib < 25:
        raise RuntimeError(
            f'Merged export needs at least 25 GiB free; found {free_gib:.1f} GiB. '
            "Choose a larger runtime disk or set EXPORT_FORMAT = 'adapter'."
        )
    print(f'Free Colab disk before merge: {free_gib:.1f} GiB.')

    adapter_config_path = TINKER_ADAPTER_DIR / 'adapter_config.json'
    if not adapter_config_path.exists():
        if TINKER_ADAPTER_DIR.exists() and any(TINKER_ADAPTER_DIR.iterdir()):
            raise RuntimeError(
                f'{TINKER_ADAPTER_DIR} is incomplete. Remove it manually before retrying.'
            )
        downloaded = weights.download(
            tinker_path=SOURCE_CHECKPOINT,
            output_dir=str(TINKER_ADAPTER_DIR),
        )
        print(f'Downloaded Tinker adapter to {downloaded}.')
    else:
        print(f'Reusing downloaded adapter at {TINKER_ADAPTER_DIR}.')

    merged_config_path = MERGED_MODEL_DIR / 'config.json'
    if not merged_config_path.exists():
        if MERGED_MODEL_DIR.exists() and any(MERGED_MODEL_DIR.iterdir()):
            raise RuntimeError(
                f'{MERGED_MODEL_DIR} is incomplete. Remove it manually before retrying.'
            )
        weights.build_hf_model(
            base_model=BASE_MODEL,
            adapter_path=str(TINKER_ADAPTER_DIR),
            output_path=str(MERGED_MODEL_DIR),
            trust_remote_code=True,
            merge_strategy='shard',
        )
        print(f'Merged model written to {MERGED_MODEL_DIR}.')
    else:
        print(f'Reusing merged model at {MERGED_MODEL_DIR}.')

    for artifact in PUBLICATION_DIR.rglob('*'):
        if artifact.is_file():
            destination = MERGED_MODEL_DIR / artifact.relative_to(PUBLICATION_DIR)
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(artifact, destination)

    required = [
        MERGED_MODEL_DIR / 'config.json',
        MERGED_MODEL_DIR / 'README.md',
        MERGED_MODEL_DIR / 'calibration_config.json',
        MERGED_MODEL_DIR / 'evaluation_summary.json',
    ]
    assert all(path.exists() for path in required)
    assert list(MERGED_MODEL_DIR.glob('*.safetensors'))
else:
    command = [
        'tinker', 'checkpoint', 'push-hf', SOURCE_CHECKPOINT,
        '--repo', HF_REPO_ID,
        '--commit-message', 'Publish CalibrateQwen unified PEFT adapter',
        '--no-model-card',
    ]
    if PUBLIC_REPO:
        command.append('--public')
    subprocess.run(command, check=True)
    print(f'PEFT adapter exported to {HF_REPO_ID}.')


## 8. Publish to Hugging Face

This cell performs the external publication. Merged mode uploads the standalone model directory. Adapter mode uploads the metadata folder after the Tinker CLI has created the PEFT repository.

Rerunning the cell creates another Hub commit with changed files. Hugging Face deduplicates unchanged large files.


In [ ]:
if EXPORT_FORMAT == 'merged':
    published_url = weights.publish_to_hf_hub(
        model_path=str(MERGED_MODEL_DIR),
        repo_id=HF_REPO_ID,
        private=not PUBLIC_REPO,
        token=os.environ['HF_TOKEN'],
    )
else:
    hf_api.upload_folder(
        folder_path=str(PUBLICATION_DIR),
        repo_id=HF_REPO_ID,
        repo_type='model',
        commit_message='Add calibration policy, evaluation report, and model card',
    )
    if PUBLIC_REPO:
        hf_api.update_repo_settings(HF_REPO_ID, private=False)
    published_url = f'https://huggingface.co/{HF_REPO_ID}'

print(f'Published: {published_url}')


## 9. Verify the published repository

We query the Hub after publication and require model weights, configuration, model card, calibration policy, output schema, and evaluation summary. This verifies the repository as a deployable artifact rather than assuming a successful upload from process exit alone.


In [ ]:
from huggingface_hub import hf_hub_download

model_info = hf_api.model_info(HF_REPO_ID, files_metadata=True)
published_files = sorted(sibling.rfilename for sibling in model_info.siblings)

required_common = {
    'README.md',
    'calibration.py',
    'calibration_config.json',
    'evaluation_summary.json',
    'structured_output_schema.json',
}
missing = required_common - set(published_files)
assert not missing, f'Missing publication files: {sorted(missing)}'

if EXPORT_FORMAT == 'merged':
    assert 'config.json' in published_files
    assert any(name.endswith('.safetensors') for name in published_files)
else:
    assert 'adapter_config.json' in published_files
    assert any(name.endswith('.safetensors') for name in published_files)

downloaded_calibration = hf_hub_download(
    HF_REPO_ID,
    'calibration_config.json',
    token=os.environ['HF_TOKEN'],
)
remote_calibration = json.loads(Path(downloaded_calibration).read_text(encoding='utf-8'))
assert math.isclose(remote_calibration['temperature'], calibration_config['temperature'])
assert math.isclose(
    remote_calibration['abstention_threshold'],
    calibration_config['abstention_threshold'],
)

print({
    'repo_id': HF_REPO_ID,
    'private': model_info.private,
    'sha': model_info.sha,
    'files': len(published_files),
    'calibration_verified': True,
})
display(pd.DataFrame({'file': published_files}))


## 10. Optional local generation smoke test

The Hub verification above checks repository completeness. Enable `RUN_LOCAL_SMOKE_TEST` to load the published model and generate one structured answer. A merged bfloat16 model needs a GPU runtime with sufficient memory. Adapter mode loads the base model and then attaches the PEFT weights.


In [ ]:
if RUN_LOCAL_SMOKE_TEST:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer_source = HF_REPO_ID if EXPORT_FORMAT == 'merged' else BASE_MODEL
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, trust_remote_code=True)
    if EXPORT_FORMAT == 'merged':
        model = AutoModelForCausalLM.from_pretrained(
            HF_REPO_ID,
            device_map='auto',
            torch_dtype='auto',
            trust_remote_code=True,
        )
    else:
        from peft import PeftModel
        base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            device_map='auto',
            torch_dtype='auto',
            trust_remote_code=True,
        )
        model = PeftModel.from_pretrained(base, HF_REPO_ID)

    system_prompt = (
        'Answer the multiple-choice question. Return one JSON object with exactly the keys '
        '"answer", "confidence", "justification", and "abstain". answer must be one uppercase '
        'option label from the prompt. confidence must be a number between 0 and 1. justification '
        'must be one concise sentence. abstain must be a JSON boolean.'
    )
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': 'Which planet is known as the Red Planet?\n\nA. Venus\nB. Mars\nC. Jupiter\nD. Mercury'},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
        )
    answer_tokens = generated[0, inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(answer_tokens, skip_special_tokens=True)
    print(generated_text)
    assert '"answer"' in generated_text
else:
    print('Local model loading skipped. Hub artifact verification is complete.')


## Release complete

A successful final verification means the Hub repository contains the selected model weights, exact calibration policy, structured-output schema, technical model card, and evaluation summary. The source Tinker path remains recorded for provenance.

For a final research release, rerun notebook 04 with `RUN_MODE = 'full'`, copy the full report to the same Drive directory, rerun this notebook, and publish the updated report metadata. The model weights remain unchanged unless the full evaluation selects a different checkpoint.
